# Shell Sort and Quicksort

```{contents}
:local:
:depth: 2
```


This supporting lesson restores the advanced algorithms from the earlier CS1 reference in the current advanced sorting chapter. Read insertion sort and recursion first. It supplements the existing searching, elementary-sort, and workload lessons.

These explanations and tasks retain the earlier CS1 teaching material.[^restored]


## Shell Sort

```{index} Shell Sort
```

Shell sort applies insertion sorting to subsequences whose elements are separated by a gap. For gap four, the sequences start at positions zero, one, two, and three, visiting every fourth element. A distant exchange can remove several inversions, moving an out-of-place value toward its destination more quickly than adjacent exchanges alone.

A single gap does not necessarily sort the whole array. Repeat with decreasing gaps, ending at one; the final pass is ordinary insertion sort over the entire array. After a gap-four pass, each gap-four subsequence is sorted, but neighboring values can still be out of order. Trace this distinction before the final pass.

The earlier lesson compared gaps 8, 4, 2, 1 with a sequence based on `h = 3*h + 1`, giving 1, 4, 13, 40, 121, and so on. Generate useful gaps for the actual array length and apply them in reverse order. The maximum gap and recurrence both affect performance; do not infer a universal complexity bound from one timing experiment. The simple halving/power-of-two family can still require quadratic work in the worst case.

### Gap Experiments

```{index} Gap Experiments
```

Trace the subsequences for gaps 4, 2, and 1 on `{9, 8, 7, 6, 5, 4, 3, 2, 1}`. Then compare `{8, 4, 2, 1}` with `{13, 4, 1}` on copies of the same longer input. Include sorted, reverse-sorted, random, duplicate-heavy, empty, and one-element arrays. Count comparisons and swaps separately. All sequences must contain a final gap one. Verify the output against a separately sorted copy before discussing efficiency.

## Quicksort

```{index} Quicksort
```

Quicksort, also called partition sort, chooses a pivot and rearranges one subrange around it. The left scan skips values smaller than the pivot; the right scan skips larger values. When the scans stop before crossing, exchange the two values and advance both positions. Advancing after a swap is essential when values equal the pivot.

The partition loop maintains these statements: everything before the left scan is at most the pivot, and everything after the right scan is at least the pivot. The unclassified interval shrinks. After the scans cross, recursively sort the remaining left and right subranges. They are not individually sorted by partitioning, but no value in the left group belongs after a value in the right group.

The same method solves smaller ranges. Empty and one-element ranges are base cases. Quicksort averages `O(n log n)` comparisons under suitable pivot/input assumptions, but repeated unbalanced partitions can require `O(n^2)` work and deep recursion. Choosing the middle **position** does not guarantee choosing the median **value**. This implementation is not stable and uses recursive stack space.

### Partition Trace

```{index} Partition Trace
```

Trace both scan indices, pivot, swaps, and recursive ranges on `{5, 2, 5, 1, 8}`. Repeat with all values equal and explain why the pointers still advance. Test subrange sorting with sentinel values outside the requested range and verify that the sentinels are unchanged. Explain why the pivot index uses `low + (high - low) / 2` rather than adding two potentially large indices.

In [11]:
using System;
using System.Collections.Generic;
static void ShellSort(int[] data, int[] descendingGaps) {
    foreach (int gap in descendingGaps) {
        if (gap <= 0) throw new ArgumentException("Gaps must be positive.");
        for (int start = 0; start < gap && start < data.Length; start++)
            for (int j = start + gap; j < data.Length; j += gap)
                for (int i = j; i >= gap && data[i] < data[i - gap]; i -= gap)
                    (data[i], data[i - gap]) = (data[i - gap], data[i]);
    }
}
static int[] Gaps(int n) {
    List<int> gaps = new List<int>();
    for (int h = 1; h < n;) {
        gaps.Add(h);
        if (h > (n - 1) / 3) break;
        h = 3 * h + 1;
    }
    gaps.Reverse(); return gaps.ToArray();
}
static void QuickSort(int[] data, int low, int high) {
    if (low >= high) return;
    int left = low, right = high, pivot = data[low + (high - low) / 2];
    while (left <= right) {
        while (data[left] < pivot) left++;
        while (data[right] > pivot) right--;
        if (left <= right) {
            (data[left], data[right]) = (data[right], data[left]);
            left++; right--;
        }
    }
    if (low < right) QuickSort(data, low, right);
    if (left < high) QuickSort(data, left, high);
}
int[] original = {9, 8, 7, 6, 5, 4, 3, 2, 1};
int[] shell = (int[])original.Clone(), quick = (int[])original.Clone();
ShellSort(shell, Gaps(shell.Length));
QuickSort(quick, 0, quick.Length - 1);
Console.WriteLine(string.Join(" ", shell));
Console.WriteLine(string.Join(" ", quick));

1 2 3 4 5 6 7 8 9
1 2 3 4 5 6 7 8 9


## Compare Correct Results

```{index} Compare Correct Results
```

The cell provides runnable implementations of the restored algorithms. ShellSort takes positive gaps in descending order; require a final one when sorting arrays of at least two elements. QuickSort takes a valid inclusive subrange, with an empty range represented by `low > high`. Add argument validation to a public-facing library wrapper rather than hiding invalid bounds inside the algorithm.

A useful experiment reports the gap sequence, input distribution, size, seed, comparison count, swap count, and correctness result. Timing can extend that evidence, but a single fast trial does not establish the asymptotic behavior. Continue with the search-performance lab for the difference between operation counts and measured elapsed time.

```{rubric} Footnotes
```

[^restored]: Restores the Shell-sort gap experiments and Quicksort partition/recursion lesson from the preserved Sorting chapter. Code includes explicit empty-range handling and an overflow-safe pivot index.